In [ ]:
#This version is a full variable load version

In [9]:
 
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
res='1km'
Np_str='1e6'

# # dx = 1 km; Np = 1M; Nt = 1 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6_1min.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6_1min.nc') #***
# res='1km'
# Np_str='1e6'


true_time=data['time']
times=data['time'].values/(1e9 * 60); times=times.astype(float);


# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))

In [10]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [ ]:
###########################################################################################################################################################################

In [ ]:
#LOADING VARIABLES
###############################################################

In [3]:
# Loading Important Variables
##############
if 'emptylike' not in globals():
    print('loading neccessary variables')
    variable='lfc'; LFC_data=data[variable].data #get w data
    variable='lcl'; LCL_data=data[variable].data #get w data
    print('done')
    empty_like=True 

loading neccessary variables
done


In [ ]:
#MAKING LAGRANGIAN BINARY ARRAY
###############################################################

In [12]:
LFC=np.zeros(parcel['z'].shape,dtype='float32')
LCL=np.zeros(parcel['z'].shape,dtype='float32')

dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
in_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min_TEST.h5'
out_file=dir2+f'LFC_LCL_binary_array_{res}_{Np_str}_5min_TEST.h5'
with h5py.File(out_file, 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('LFC', data=LFC) #binary array for general updraft (w>=0.1)
    f.create_dataset('LCL', data=LCL) #binary array for general updraft (w>=0.5 & qc+qi>=1e-6)
    

with h5py.File(in_file, 'r') as f_in, h5py.File(out_file, 'r+') as f_out:

    #READING BACK DATA LATER
    parcel_z = f_in['z'][:]
    Y = f_in['Y'][:]
    X = f_in['X'][:]
    
    Nt=len(data['time'])
    Np=len(parcel['xh'])
    for p in np.arange(Np):
        if np.mod(p,25e3)==0: print(f"{p}/{len(parcel['xh'])}")
    
        #Get Indicies
        zs=parcel_z[:,p]
        ys=Y[:,p]
        xs=X[:,p]
        ts = np.arange(Nt)  
    
        #Get Values
        lfcs = LFC_data[ts, ys, xs]
        lcls = LCL_data[ts, ys, xs]
        
        f_out['LFC'][:,p]=(zs>=lfcs)*1
        f_out['LCL'][:,p]=(zs>=lcls)*1

0/1000000
1000/1000000
2000/1000000
3000/1000000
4000/1000000
5000/1000000
6000/1000000
7000/1000000
8000/1000000
9000/1000000
10000/1000000
11000/1000000
12000/1000000
13000/1000000
14000/1000000
15000/1000000
16000/1000000
17000/1000000
18000/1000000
19000/1000000
20000/1000000
21000/1000000
22000/1000000
23000/1000000
24000/1000000
25000/1000000
26000/1000000
27000/1000000
28000/1000000
29000/1000000
30000/1000000
31000/1000000
32000/1000000
33000/1000000
34000/1000000
35000/1000000
36000/1000000
37000/1000000
38000/1000000
39000/1000000
40000/1000000
41000/1000000
42000/1000000
43000/1000000
44000/1000000
45000/1000000
46000/1000000


KeyboardInterrupt: 

In [ ]:
#########################################

In [21]:
# # Reading Back Data Later
# ##############
# import h5py
# dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# with h5py.File(dir2+f'LFC_LCL_binary_array.h5', 'r') as f:
#     # Load the dataset by its name
#     LFC = f['LFC'][:]
#     LCL = f['LCL'][:]